# FusionTrack — step‑by‑step single object track (pedagogical)

This notebook is **for building intuition**, not a polished demo. It re‑implements a **tiny slice** of the pipeline so you can see **numbers at each time step**: what the state is, when `predict` runs, when `update_*` ingests a measurement, and what happens in the **occlusion** window (frames 40–60) where the camera is forced off and radar must carry the track.

Run the project root as your working directory, or add the parent folder to `PYTHONPATH` (the first code cell does that defensively).


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
from numpy.random import default_rng

from src import camera_sim, ekf, radar_sim, utils
from src.fusion import (
    FUSION_FRAME_DT_S,
    _initial_state_from_ground_truth,
    _make_smooth_curved_trajectory,
    run_fusion_demo,
    RADAR_MEASUREMENT_VAR_M2,
)


## Frame 0 — initial state and what **P** (covariance) means here

The `KFTracker` in this project uses a **4×4** state covariance $P$ for $[x, y, v_x, v_y]$ (linear KF, not a native polar EKF — see the README naming section).

- The **2×2 top‑left block** is the **marginal** covariance of **position** in the world plane. Its eigenvectors point along the *principal axes* of the “error ellipse”. Before any measurement at time 0, you should expect **large** diagonal entries in that block: we *do not* assume we know the target to millimeters.
- The **2×2 bottom‑right block** governs *velocity* uncertainty. If you *badly* initialize velocity, the *predicted* position uncertainty grows faster, because a wrong heading or speed is integrated forward by the constant‑velocity $F$ matrix.

Below we print `P` for a freshly created filter so you can *read* the numbers while you look at the source (`src/ekf.py`).


In [ ]:
SEED = 7
rng = default_rng(SEED)
true_xy = _make_smooth_curved_trajectory(100)
x0 = _initial_state_from_ground_truth(true_xy, FUSION_FRAME_DT_S, rng)
r_cam = utils.pixel_noise_to_world_covariance(
    camera_sim.CAMERA_CENTER_NOISE_STD_PX,
    camera_sim.CAMERA_CENTER_NOISE_STD_PX,
)
r_radar = np.eye(2) * RADAR_MEASUREMENT_VAR_M2  # ~2 m 1-σ per axis, tighter than camera (~4 m)
tr0 = ekf.KFTracker(x0, dt=FUSION_FRAME_DT_S, r_camera=r_cam, r_radar=r_radar)
print("Initial state [x, y, vx, vy]:", tr0.get_state())
print("Initial covariance P (4x4) — look at 2x2 top-left (position) block first:")
print(np.array2string(tr0.get_covariance(), precision=2, suppress_small=True))


## `P` at frames 1 and 2 — after `predict` vs after each `update`

`trace(P)` (full $4 \times 4$) is a blunt summary: it **often** rises right after `predict` (process noise $Q$ added) and **falls** when an informative `update_camera` / `update_radar` runs. In interviews, you usually narrate the **2×2 position block** first, then velocity.

The RNG here is a fresh walk (`rng_p`) for clarity — compare the **pattern** of tr(P) to the “Frames 1–5” cell, not the bit-identical values (that cell reuses a single `rng` stream from earlier).

In [ ]:
r_radar_nb = np.eye(2) * RADAR_MEASUREMENT_VAR_M2
x_w = _initial_state_from_ground_truth(true_xy, FUSION_FRAME_DT_S, default_rng(7))
trp = ekf.KFTracker(x_w, dt=FUSION_FRAME_DT_S, r_camera=r_cam, r_radar=r_radar_nb)
rng_p = default_rng(7)
for k in (1, 2):
    trp.predict()
    print(f"\n======== frame {k} — AFTER predict (time update) ========")
    P = trp.get_covariance()
    print("tr(P) =", float(np.trace(P)))
    print(P)
    cam = camera_sim.camera_measurement_from_truth(
        float(true_xy[k, 0]), float(true_xy[k, 1]), k, rng_p
    )
    rad = radar_sim.radar_measurement_from_true_position(
        float(true_xy[k, 0]), float(true_xy[k, 1]), rng_p
    )
    if cam is not None:
        wx, wy = utils.pixel_to_world_meters(cam.u_center, cam.v_center)
        trp.update_camera(np.array([wx, wy], dtype=np.float64), r_override=r_cam)
        print("--- after camera update, tr(P) =", float(np.trace(trp.get_covariance())))
    if rad is not None:
        zr = np.array(radar_sim.polar_to_world_xy(rad), dtype=np.float64)
        trp.update_radar(zr, r_override=r_radar_nb)
    print("--- end of step, tr(P) =", float(np.trace(trp.get_covariance())))
    print(trp.get_covariance())

## Frames 1 through 5 — **predict** then **update** manually, printed each step

For each `k` in `{1, 2, 3, 4, 5}` we will:
1. Call `predict()` — time‑align the Gaussian through the **constant velocity** $F$ and add $Q$ (process noise widens the predicted covariance; # INTERVIEW CRITICAL: this is where “how maneuverable is the model?” is encoded).
2. Ask the *same* simulators the main pipeline uses to give us a camera and radar measurement at frame `k` (either may be `None`).
3. If both exist, update **camera first**, then **radar** (sequential updates — not the *joint* one‑shot update; good enough for this portfolio, but know the difference in an interview … see Bar‑Shalom, sequential vs joint measurement update).
4. Print the new posterior $x$ after all updates for that step.


In [ ]:
x = x0.copy()
tr = ekf.KFTracker(x, dt=FUSION_FRAME_DT_S, r_camera=r_cam, r_radar=r_radar)
for k in range(1, 6):
    tr.predict()
    # Camera path
    cam = camera_sim.camera_measurement_from_truth(
        float(true_xy[k, 0]), float(true_xy[k, 1]), k, rng
    )
    rad = radar_sim.radar_measurement_from_true_position(
        float(true_xy[k, 0]), float(true_xy[k, 1]), rng
    )
    if cam is not None:
        wx, wy = utils.pixel_to_world_meters(cam.u_center, cam.v_center)
        zc = np.array([wx, wy], dtype=np.float64)
    else:
        zc = None
    if rad is not None:
        zx, zy = radar_sim.polar_to_world_xy(rad)
        zr = np.array([zx, zy], dtype=np.float64)
    else:
        zr = None
    print(f"\n--- frame {k} ---")
    print(f"  camera meas world: {zc if zc is not None else 'MISS'}")
    print(f"  radar meas world: {zr if zr is not None else 'MISS'}")
    if zc is not None and zr is not None:
        tr.update_camera(zc, r_override=r_cam)
        tr.update_radar(zr, r_override=r_radar)
    elif zc is not None:
        tr.update_camera(zc, r_override=r_cam)
    elif zr is not None:
        tr.update_radar(zr, r_override=r_radar)
    else:
        print("  (no measurement — only prediction step carried forward)")
    print("  posterior [x, y, vx, vy]:", tr.get_state())


## Frame 40 — **occlusion** begins (camera forced off, radar can still return)

The camera simulator in `src/camera_sim.py` returns `None` for *every* frame in **40…60** inclusive. In that window the **fused** filter is running **open‑loop in the visual subspace** — only radar updates nudge the position estimate when a radar return exists.

What you should watch for: **larger** trace of the $2\times2$ position covariance block, or visually **wider** error ellipses, because we have lost a relatively tight measurement mode (the camera) that keeps lateral error small when it is healthy.


In [ ]:
res = run_fusion_demo(rng=default_rng(7))
k = 40
print(f"At frame {k} (first frame of the occlusion block), camera is always None: {res['camera_pixel_centers'][k]!r}")
print("Ground truth @40:          ", res["ground_truth"][k])
print("Fused position @40:      ", res["fused"][k])
print("Radar return @40:       ", res["radar_polars"][k])  # may be None (miss) or a PolarRadarReturn
print("Cov trace tr(P) @40 (scalar summary):", float(res["cov_trace"][k]))


### Same run, adjacent frames (39 → 40 → 41)
Confirms the camera drops exactly when the sim says it should, while radar can still return.


In [ ]:
res = run_fusion_demo(rng=default_rng(7))
for j in (39, 40, 41):
    print(f"\n--- frame {j} ---")
    print("  camera:", res["camera_pixel_centers"][j])
    print("  fused: ", res["fused"][j])
    print("  tr(P): ", res["cov_trace"][j])


## Frame 60 — end of the occlusion **window**; camera *can* return

At or after frame 61 the synthetic camera re‑enters the information stack and the fused estimate should **tighten** back up, assuming the KF was not so badly misfit during the radar‑only run that a single camera frame cannot pull it in.


In [ ]:
k60 = 60
res = run_fusion_demo(rng=default_rng(7))
print(f"At frame {k60}, camera is back (first frame after the inclusive dropout window is 61 — check sim constants).")
print(f"Frame 60 (last occluded) camera: {res['camera_pixel_centers'][k60]}")
k61 = 61
print(f"Frame {k61} camera: {res['camera_pixel_centers'][k61]}")
print(f"Fused {k60} -> {k61} delta:", res['fused'][k61] - res['fused'][k60])


## Putting it all together — inline plots

The two *portfolio* figures are produced by `plot_results()` in `src/fusion.py`.
Running them inline here shows the **money shot** without having to leave the notebook:

* **Figure 1** — trajectory comparison: EKF fused (polar radar) in red, KF fused (Cartesian radar) in blue dashed, camera-only and radar-only baselines in dotted lines, 95% uncertainty ellipses every 10 frames drawn for the EKF.
* **Figure 2** — per-frame L2 error for all four tracks with the occlusion window (frames 40–60) shaded grey.  During occlusion, the fused tracks lean on radar only; the EKF error should be ≤ the KF error because the polar *R* is physically better calibrated.

`tr(P)` (trace of the full 4×4 covariance) is shown in the third panel below — it jumps during the occlusion window and settles again when the camera returns.


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from src.fusion import plot_results, run_fusion_demo
from src.camera_sim import CAMERA_OCCLUSION_START_FRAME, CAMERA_OCCLUSION_END_FRAME

%matplotlib inline

res = run_fusion_demo(rng=default_rng(7))
fig1, fig2 = plot_results(res, show=False)
plt.figure(fig1.number)
plt.tight_layout()
plt.show()
plt.figure(fig2.number)
plt.tight_layout()
plt.show()

# --- tr(P) panel — covariance summary over time ---
fig3, ax3 = plt.subplots(figsize=(8, 3))
frames = range(len(res["cov_trace"]))
ax3.plot(frames, res["cov_trace"],    color="C0", linestyle="--", label="KF fused  tr(P)")
ax3.plot(frames, res["ekf_cov_trace"], color="C3", linewidth=1.6, label="EKF fused tr(P)")
ax3.axvspan(CAMERA_OCCLUSION_START_FRAME, CAMERA_OCCLUSION_END_FRAME,
            color="0.4", alpha=0.15, label="Camera occlusion")
ax3.set_xlabel("Frame k")
ax3.set_ylabel("tr(P) — scalar covariance proxy")
ax3.set_title("Covariance trace over time — rises during occlusion, falls when camera returns")
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("KF  tr(P) mean over occlusion  frames 40-60:", res['cov_trace'][40:61].mean().round(2))
print("EKF tr(P) mean over occlusion  frames 40-60:", res['ekf_cov_trace'][40:61].mean().round(2))
